# SE-LLM-350M — Kaggle TPU Pre-Training Notebook

**Instructions:**
1. Enable TPU: Settings → Accelerator → **TPU v5e-8**
2. Add your dataset: `se-llm-data` (containing train.bin, val.bin)
3. Add Kaggle Secrets: `WANDB_API_KEY` and `GITHUB_TOKEN`
4. Click **Save Version** → **Save & Run All (Commit)** — training starts automatically

> 💡 **One session is enough!** Training 7B tokens on TPU v5e-8 takes ~4–8 hours.
> You have a 9-hour session limit, so the full pre-training completes in a single run.

In [ ]:
# ── Cell 1: Verify TPU ────────────────────────────────────────

import os
os.environ.pop('CLOUD_TPU_TASK_ID', None)
os.environ.pop('TPU_PROCESS_ADDRESSES', None)

import torch
print(f'PyTorch: {torch.__version__}')

try:
    import torch_xla
    import torch_xla.runtime as xr
    device = torch_xla.device()
    print(f'✅ TPU available: {device}')
    print(f'   World size (chips): {xr.world_size()}')
except ImportError:
    print('⚠️  torch_xla not found — make sure accelerator is set to TPU v5e-8')
    if torch.cuda.is_available():
        print(f'   Falling back to GPU: {torch.cuda.get_device_name(0)}')
        print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    else:
        print('   Falling back to CPU')

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────
# Note: torch_xla is pre-installed on Kaggle TPU instances.
# We only need to install the remaining Python packages.
!pip uninstall -y tensorflow && pip install -q tensorflow-cpu
!pip install -q wandb tokenizers datasets pyyaml rich

In [ ]:
# ── Cell 3: Clone training code from GitHub ───────────────────
import os
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

GITHUB_REPO = 'Abhik2005/se-llm-data'  # ← UPDATE THIS

if not os.path.exists('/kaggle/working/se-llm-350m'):
    # Use token for private repo access
    try:
        token = secrets.get_secret('GITHUB_TOKEN')
        clone_url = f'https://{token}@github.com/{GITHUB_REPO}.git'
    except Exception:
        clone_url = f'https://github.com/{GITHUB_REPO}.git'
    
    !git clone {clone_url} /kaggle/working/se-llm-350m
    print('Repo cloned')
else:
    !git -C /kaggle/working/se-llm-350m pull
    print('Repo updated')

%cd /kaggle/working/se-llm-350m
!ls -la

In [ ]:
# ── Cell 4: Link dataset files ────────────────────────────────
import os

os.makedirs('data/processed', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

# Link train.bin from Kaggle dataset
DATASET_PATH = '/kaggle/input/se-llm-data'  # ← your Kaggle dataset name

for fname in ['train.bin', 'val.bin']:
    src  = f'{DATASET_PATH}/{fname}'
    dest = f'data/processed/{fname}'
    if os.path.exists(src) and not os.path.exists(dest):
        os.symlink(src, dest)
        print(f'Linked: {dest} → {src}')
    elif os.path.exists(dest):
        print(f'Already linked: {dest}')
    else:
        print(f'WARNING: {src} not found — check dataset name')

# Link tokenizer
os.makedirs('tokenizer', exist_ok=True)
tok_src = f'{DATASET_PATH}/tokenizer.json'
tok_dst = 'tokenizer/tokenizer.json'
if os.path.exists(tok_src) and not os.path.exists(tok_dst):
    os.symlink(tok_src, tok_dst)
    print(f'Linked tokenizer')

# Check sizes
for f in ['data/processed/train.bin', 'data/processed/val.bin']:
    if os.path.exists(f):
        size_gb = os.path.getsize(f) / 1e9
        print(f'{f}: {size_gb:.2f} GB')

In [ ]:
# ── Cell 5: Login to W&B ──────────────────────────────────────
import wandb
from kaggle_secrets import UserSecretsClient

try:
    secrets = UserSecretsClient()
    wandb_key = secrets.get_secret('WANDB_API_KEY')
    wandb.login(key=wandb_key)
    print('W&B logged in')
except Exception as e:
    print(f'W&B login failed: {e} — training will continue without logging')

In [ ]:
# ── Cell 6: START TRAINING ────────────────────────────────────
# train.py uses xmp.spawn to launch 8 parallel workers automatically
# (one per TPU chip). You do not need to do anything special.
#
# --resume auto: continues from latest checkpoint if one exists
# --resume none: starts from scratch

!PYTHONUNBUFFERED=1 python training/train.py \
    --config configs/350m.yaml \
    --resume auto

In [ ]:
# ── Cell 7: Save checkpoints to output ────────────────────────
# Kaggle saves /kaggle/working/ automatically on Commit.
# This cell also copies checkpoints to /kaggle/working/output
# for extra safety and easy access from the next session.

import shutil, glob, os

output_dir = '/kaggle/working/output'
os.makedirs(output_dir, exist_ok=True)

for ckpt in glob.glob('checkpoints/*.pt'):
    dest = f'{output_dir}/{os.path.basename(ckpt)}'
    shutil.copy(ckpt, dest)
    print(f'Saved: {dest}')

# Show final checkpoint info
latest = 'checkpoints/latest.pt'
if os.path.exists(latest):
    import torch
    ckpt = torch.load(latest, map_location='cpu', weights_only=False)
    print(f'\nFinal checkpoint:')
    print(f'  Step:     {ckpt["step"]:,}')
    print(f'  Tokens:   {ckpt["tokens_processed"]/1e9:.3f}B')
    print(f'  Val Loss: {ckpt["val_loss"]:.4f}')

    pct = 100 * ckpt['tokens_processed'] / 7_000_000_000
    print(f'  Progress: {pct:.1f}% of 7B tokens complete')